# Transit geometry sampling: `cosi`, impact parameter $b$, or normalized $\beta$?

This notebook derives

$$\beta \equiv \frac{b_\mathrm{primary}}{1+k}, \qquad k=R_p/R_\star,$$

and tests whether it is a better sampling coordinate than the ordinary impact parameter $b$ or $\cos i$. The answer is deliberately not assumed in advance: we compare support efficiency, induced priors, and MCMC efficiency under a common physical posterior. We also compare with the published Espinoza (2018) $(r_1,r_2)$ mapping.

> **Executive summary.** $\beta$ is convenient for a fit explicitly conditioned on a known transit because its support is always $[0,1]$. It is not universally superior: it encodes a different joint prior from a prior uniform in $(b,k)$, and $\cos i$ remains the natural coordinate for an unconditioned isotropic-orientation prior.

## 1. Derivation

allesfitter uses

$$s \equiv r_\mathrm{suma}=\frac{R_\star+R_p}{a}, \qquad k=\frac{R_p}{R_\star}.$$

Therefore $R_\star/a=s/(1+k)$. At primary conjunction, using the same eccentricity approximation as `impact_parameters_smart`,

$$b_\mathrm{primary}=\frac{a\cos i}{R_\star}\frac{1-e^2}{1+e\sin\omega}
=\frac{1+k}{s}\cos i\,C,$$

where $C=(1-e^2)/(1+e\sin\omega)$. A primary transit requires

$$|b_\mathrm{primary}|<1+k.$$

Dividing by the moving grazing boundary gives

$$\boxed{\beta=\frac{b_\mathrm{primary}}{1+k}=\frac{\cos i}{s}C}, \qquad 0\leq\beta<1,$$

and the inverse transformation is

$$\boxed{\cos i=\beta s\frac{1+e\sin\omega}{1-e^2}}.$$

For a circular orbit, $\beta=\cos i/s$. Notice that $k$ cancels: this is why the transit boundary becomes rectangular in $\beta$.

In [1]:
import warnings

import emcee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(20260715)

K_BOUNDS = (0.01, 0.20)
S_BOUNDS = (0.03, 0.15)
MU_MAX = 0.20  # broad independent cos(i) bound
B_MAX = 1 + K_BOUNDS[1]


def eccentricity_correction(e, omega):
    return (1 - e**2) / (1 + e * np.sin(omega))


def beta_from_cosi(cosi, rsuma, e=0.0, omega=0.0):
    return cosi * eccentricity_correction(e, omega) / rsuma


def cosi_from_beta(beta, rsuma, e=0.0, omega=0.0):
    return beta * rsuma / eccentricity_correction(e, omega)


# Round-trip checks, including eccentric cases.
trial_beta = rng.uniform(0, 1, 10_000)
trial_s = rng.uniform(*S_BOUNDS, 10_000)
trial_e = rng.uniform(0, 0.6, 10_000)
trial_omega = rng.uniform(-np.pi, np.pi, 10_000)
trial_cosi = cosi_from_beta(trial_beta, trial_s, trial_e, trial_omega)
np.testing.assert_allclose(beta_from_cosi(trial_cosi, trial_s, trial_e, trial_omega), trial_beta)
print("Eccentric beta <-> cos(i) round trip passed.")

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (60)

Eccentric beta <-> cos(i) round trip passed.


## 2. Experiment A — how much rectangular prior volume is invalid?

We independently draw broad rectangular priors in each coordinate. This measures rejection caused by the coordinate bounds alone; it is not yet a posterior comparison.

- `cosi`: draw $s$ and $\cos i$ independently; for circular orbits require $\cos i<s$.
- `b`: draw $k$ and $b$ independently; require $b<1+k$.
- `beta`: draw $\beta\in[0,1]$; every draw transits by construction.

In [2]:
N = 300_000
k = rng.uniform(*K_BOUNDS, N)
s = rng.uniform(*S_BOUNDS, N)
mu = rng.uniform(0, MU_MAX, N)
b = rng.uniform(0, B_MAX, N)
beta = rng.uniform(0, 1, N)

valid_cosi = mu < s
valid_b = b < 1 + k
valid_beta = np.ones(N, dtype=bool)

support = pd.DataFrame(
    {
        "valid fraction": [valid_cosi.mean(), valid_b.mean(), valid_beta.mean()],
        "wasted fraction": [1 - valid_cosi.mean(), 1 - valid_b.mean(), 0.0],
    },
    index=["cosi rectangle", "b rectangle", "beta rectangle"],
)
display(support.style.format("{:.1%}"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
take = rng.choice(N, 12_000, replace=False)
axes[0].scatter(s[take], mu[take], c=valid_cosi[take], s=2, cmap="coolwarm_r", alpha=0.5)
axes[0].plot(S_BOUNDS, S_BOUNDS, "k--", label=r"$\cos i=s$")
axes[0].set(xlabel="rsuma s", ylabel="cosi", title="Independent cosi bounds")
axes[0].legend()
axes[1].scatter(k[take], b[take], c=valid_b[take], s=2, cmap="coolwarm_r", alpha=0.5)
kg = np.linspace(*K_BOUNDS, 100)
axes[1].plot(kg, 1 + kg, "k--", label=r"$b=1+k$")
axes[1].set(xlabel="radius ratio k", ylabel="impact parameter b", title="Independent b bounds")
axes[1].legend()
axes[2].scatter(s[take], beta[take], s=2, alpha=0.35)
axes[2].set(xlabel="rsuma s", ylabel=r"$\beta$", title=r"$\beta$ support is rectangular")
plt.tight_layout()

,valid fraction,wasted fraction
cosi rectangle,45.1%,54.9%
b rectangle,92.1%,7.9%
beta rectangle,100.0%,0.0%


<Figure size 1400x400 with 3 Axes>

The valid fraction depends on the chosen bounds. Direct $b$ can also be made 100% valid by drawing $b=(1+k)u$ with $u\sim U(0,1)$—but that $u$ is exactly $\beta$. Thus the support advantage is real, but not unique to the symbol or implementation.

## 3. The important caveat: these coordinates can imply different priors

A hard transit cut applied to an isotropic prior $\cos i\sim U(0,1)$ does more than remove invalid samples. For fixed $(s,e,\omega)$, the permitted interval has width

$$\cos i_\max=s\frac{1+e\sin\omega}{1-e^2}.$$

Consequently, conditioning by rejection weights the retained distribution of $(s,e,\omega)$ by the geometric transit probability. Drawing $\beta\sim U(0,1)$ independently instead preserves the supplied priors on $(s,e,\omega)$. These correspond to different scientific questions:

- isotropic `cosi` plus transit selection: model the population and its probability of transiting;
- uniform `beta`: condition the analysis on the fact that this particular system is already known to transit.

In [3]:
N_PRIOR = 500_000
s_parent = rng.uniform(*S_BOUNDS, N_PRIOR)
mu_parent = rng.uniform(0, 1, N_PRIOR)
s_selected = s_parent[mu_parent < s_parent]
s_conditional = rng.uniform(*S_BOUNDS, len(s_selected))

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(*S_BOUNDS, 45)
ax.hist(s_parent, bins=bins, density=True, histtype="step", lw=2, label="parent prior on s")
ax.hist(
    s_selected,
    bins=bins,
    density=True,
    histtype="step",
    lw=2,
    label="isotropic cosi + transit selection",
)
ax.hist(
    s_conditional, bins=bins, density=True, histtype="step", lw=2, label="independent uniform beta"
)
ax.set(xlabel="rsuma s", ylabel="density", title="Transit selection changes the geometry prior")
ax.legend()
print(
    f"Mean s: parent={s_parent.mean():.4f}, selected={s_selected.mean():.4f}, beta-conditioned={s_conditional.mean():.4f}"
)

Mean s: parent=0.0900, selected=0.1033, beta-conditioned=0.0902


<Figure size 700x400 with 1 Axes>

## 4. Experiment B — MCMC efficiency under the same physical posterior

We now compare coordinates fairly. All three samplers target the same prior, uniform in $(k,s,\beta)$, and the same toy transit likelihood. Jacobians are included when sampling in $b$ or `cosi`:

$$\beta=\frac{b}{1+k} \Rightarrow \left|\frac{d\beta}{db}\right|=\frac{1}{1+k},$$

$$\beta=\frac{\cos i}{s} \Rightarrow \left|\frac{d\beta}{d\cos i}\right|=\frac{1}{s}.$$

The toy data constrain depth $k^2$, a duration-like chord observable $q=s\sqrt{1-\beta^2}$, and $s$ through a stellar-density-like measurement. This is not a replacement for a light-curve injection test; it isolates geometry-coordinate behavior cheaply and reproducibly.

In [4]:
TRUTH = np.array([0.070, 0.095, 0.55])  # k, s, beta
OBS = {
    "depth": (TRUTH[0] ** 2, 3.0e-4),
    "chord": (TRUTH[1] * np.sqrt(1 - TRUTH[2] ** 2), 0.0025),
    "s_density": (TRUTH[1], 0.008),
}


def physical_loglike(k, s, beta):
    if not (K_BOUNDS[0] < k < K_BOUNDS[1] and S_BOUNDS[0] < s < S_BOUNDS[1] and 0 < beta < 1):
        return -np.inf
    depth = k**2
    chord = s * np.sqrt(1 - beta**2)
    terms = [
        ((depth - OBS["depth"][0]) / OBS["depth"][1]) ** 2,
        ((chord - OBS["chord"][0]) / OBS["chord"][1]) ** 2,
        ((s - OBS["s_density"][0]) / OBS["s_density"][1]) ** 2,
    ]
    return -0.5 * sum(terms)


def decode(theta, coordinate):
    k, s, g = theta
    if coordinate == "beta":
        beta = g
        log_jac = 0.0
    elif coordinate == "b":
        beta = g / (1 + k)
        log_jac = -np.log1p(k)  # d beta / d b
    elif coordinate == "cosi":
        beta = g / s
        log_jac = -np.log(s)  # d beta / d cosi
    else:
        raise ValueError(coordinate)
    return k, s, beta, log_jac


def log_probability(theta, coordinate):
    k, s, beta, log_jac = decode(theta, coordinate)
    value = physical_loglike(k, s, beta)
    return value + log_jac if np.isfinite(value) else -np.inf


def encode(physical, coordinate):
    k, s, beta = np.asarray(physical).T
    geometry = {"beta": beta, "b": beta * (1 + k), "cosi": beta * s}[coordinate]
    return np.column_stack([k, s, geometry])


def physical_chain(chain, coordinate):
    flat = chain.reshape(-1, 3)
    decoded = np.array([decode(row, coordinate)[:3] for row in flat])
    return decoded.reshape(chain.shape)


def initial_walkers(seed, nwalkers=48):
    r = np.random.default_rng(seed)
    p = np.column_stack(
        [
            np.clip(r.normal(TRUTH[0], 0.008, nwalkers), *K_BOUNDS),
            np.clip(r.normal(TRUTH[1], 0.008, nwalkers), *S_BOUNDS),
            np.clip(r.normal(TRUTH[2], 0.07, nwalkers), 0.02, 0.98),
        ]
    )
    return p


def run_one(coordinate, seed, nsteps=4000, burn=1000):
    np.random.seed(seed)  # emcee's proposal RNG
    p0 = encode(initial_walkers(seed), coordinate)
    sampler = emcee.EnsembleSampler(
        len(p0),
        3,
        log_probability,
        args=(coordinate,),
    )
    sampler.run_mcmc(p0, nsteps, progress=False)
    chain = sampler.get_chain(discard=burn)
    pchain = physical_chain(chain, coordinate)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tau = np.asarray(emcee.autocorr.integrated_time(pchain, tol=0, quiet=True)).reshape(-1)
    nsamples = np.prod(pchain.shape[:2])
    result = {
        "coordinate": coordinate,
        "seed": seed,
        "acceptance": sampler.acceptance_fraction.mean(),
        "tau_k": tau[0],
        "tau_s": tau[1],
        "tau_beta": tau[2],
        "min_ESS": nsamples / tau.max(),
    }
    return result, pchain


records, example_chains = [], {}
for coordinate in ("cosi", "b", "beta"):
    for seed in (11, 22, 33):
        record, chain = run_one(coordinate, seed)
        records.append(record)
        if seed == 11:
            example_chains[coordinate] = chain

runs = pd.DataFrame(records)
summary = (
    runs.groupby("coordinate")
    .median(numeric_only=True)[["acceptance", "tau_k", "tau_s", "tau_beta", "min_ESS"]]
    .sort_values("tau_beta")
)
display(
    summary.style.format(
        {
            "acceptance": "{:.3f}",
            "tau_k": "{:.1f}",
            "tau_s": "{:.1f}",
            "tau_beta": "{:.1f}",
            "min_ESS": "{:.0f}",
        }
    )
)
best = summary["tau_beta"].idxmin()
print(f"Lowest median beta autocorrelation time in this experiment: {best!r}.")

,acceptance,tau_k,tau_s,tau_beta,min_ESS
coordinate,,,,,
cosi,0.609,37.9,46.6,48.6,2964
b,0.594,39.6,53.5,56.9,2529
beta,0.594,36.8,54.4,59.6,2417


Lowest median beta autocorrelation time in this experiment: 'cosi'.


In [6]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
colors = {"cosi": "C3", "b": "C1", "beta": "C0"}
for col, coordinate in enumerate(("cosi", "b", "beta")):
    chain = example_chains[coordinate]
    axes[0, col].plot(chain[:, :8, 2], alpha=0.35, lw=0.7, color=colors[coordinate])
    axes[0, col].axhline(TRUTH[2], color="k", ls="--")
    axes[0, col].set(title=f"sampled via {coordinate}", ylabel=r"derived $\beta$")
    flat = chain.reshape(-1, 3)[::20]
    axes[1, col].scatter(flat[:, 1], flat[:, 2], s=2, alpha=0.15, color=colors[coordinate])
    axes[1, col].plot(TRUTH[1], TRUTH[2], "k*", ms=10)
    axes[1, col].set(xlabel="rsuma s", ylabel=r"$\beta$")
plt.tight_layout()

<Figure size 1400x700 with 6 Axes>

Interpret the table empirically: lower autocorrelation time and higher effective sample size are better. With the fixed seeds and dependency versions used to validate this notebook, direct `cosi` has the lowest median $\beta$ autocorrelation time, while $\beta$ has the only completely rectangular naive support. This is a useful counterexample to the claim that $\beta$ must always sample better. Because `emcee` is affine-invariant and the toy posterior lies away from the grazing boundary, its strongest advantage here is guaranteed support and clearer conditional-prior semantics—not faster posterior mixing.

## 5. Comparison with the published Espinoza (2018) mapping

Espinoza (2018, RNAAS 2, 209; [arXiv:1811.04859](https://arxiv.org/abs/1811.04859)) maps $(r_1,r_2)\in[0,1]^2$ into the valid $(b,k)$ polygon and samples it uniformly. This is the literature-backed solution implemented by `juliet`. It is not the same prior as independent uniform $(\beta,k)$:

$$b=(1+k)\beta, \qquad db=(1+k)d\beta.$$

Thus uniform area in $(b,k)$ corresponds to $p(\beta,k)\propto1+k$, whereas independent uniform $(\beta,k)$ gives $p(b,k)\propto1/(1+k)$. For planetary $k\ll1$ the numerical difference can be small, but it is conceptually real.

In [7]:
def espinoza_r1r2_to_bk(r1, r2, k_lo=K_BOUNDS[0], k_hi=K_BOUNDS[1]):
    r1, r2 = np.broadcast_arrays(r1, r2)
    area_ratio = (k_hi - k_lo) / (2 + k_lo + k_hi)
    b = np.empty_like(r1, dtype=float)
    k = np.empty_like(r1, dtype=float)
    triangle = r1 <= area_ratio
    rectangle = ~triangle
    b[rectangle] = (1 + k_lo) * (1 + (r1[rectangle] - 1) / (1 - area_ratio))
    k[rectangle] = (1 - r2[rectangle]) * k_lo + r2[rectangle] * k_hi
    root = np.sqrt(r1[triangle] / area_ratio)
    b[triangle] = (1 + k_lo) + root * r2[triangle] * (k_hi - k_lo)
    k[triangle] = k_hi + (k_lo - k_hi) * root * (1 - r2[triangle])
    return b, k


r1 = rng.uniform(0, 1, N)
r2 = rng.uniform(0, 1, N)
b_esp, k_esp = espinoza_r1r2_to_bk(r1, r2)
beta_esp = b_esp / (1 + k_esp)
assert np.all((b_esp >= 0) & (b_esp <= 1 + k_esp))

k_ind = rng.uniform(*K_BOUNDS, N)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
idx = rng.choice(N, 20_000, replace=False)
axes[0].scatter(k_esp[idx], b_esp[idx], s=2, alpha=0.2)
axes[0].plot(kg, 1 + kg, "k--")
axes[0].set(xlabel="k", ylabel="b", title="Espinoza: uniform valid (b, k) area")
axes[1].hist(k_esp, bins=50, density=True, histtype="step", lw=2, label="Espinoza r1,r2")
axes[1].hist(
    k_ind, bins=50, density=True, histtype="step", lw=2, label="independent uniform beta, k"
)
axes[1].set(
    xlabel=r"radius ratio $k$",
    ylabel="density",
    title="Different joint priors induce different k marginals",
)
axes[1].legend()
plt.tight_layout()

<Figure size 1100x400 with 2 Axes>

## 6. Conclusions

| Scientific/sampling goal | Recommended coordinate | Reason |
|---|---|---|
| Fit a particular system already known to transit; want a uniform conditional impact-location prior | $\beta\sim U(0,1)$ | Rectangular support, simple eccentric inverse transform, preserves supplied geometry priors |
| Want a literature-backed prior uniform over the physically valid $(b,k)$ area | Espinoza $(r_1,r_2)$ | Exact joint mapping used by `juliet`; no rejection |
| Model an underlying isotropic population or include geometric transit probability in evidence | $\cos i\sim U(0,1)$ plus the transit selection function | Retains the correct solid-angle measure and selection weighting |
| Sample direct $b$ | Acceptable with a conditional transform or Espinoza mapping | Naive independent bounds have a moving $1+k$ boundary |

So $\beta$ is **superior for support and often for sampling convenience**, but it is **not universally statistically superior**. The choice is a prior-model decision. For allesfitter's confirmed-transit use case, $\beta$ is a defensible future coordinate; for a published general-purpose implementation, the Espinoza $(r_1,r_2)$ mapping is the stronger precedent.

### Limitations and next experiment

This notebook uses an analytic toy likelihood to isolate parameter geometry. Before changing allesfitter's production parameterization, repeat the comparison with injected `ellc` light curves across central, grazing, low-S/N, and eccentric cases, using both `emcee` and the configured nested samplers. Compare posterior calibration and log-evidence as well as runtime and ESS.